# Volet 1 — Nettoyage intelligent et classification des réclamations
### Notebook d'illustration — diagnostic, nettoyage, visualisations

Ce notebook rejoue les étapes du pipeline (`pipeline/step0` à `step5`) sur les
vraies données, avec des graphiques pensés pour être capturés en image et
insérés directement dans le rapport ou la présentation.

Chaque section correspond à un chapitre du rapport (indiqué entre crochets).


In [ ]:
%matplotlib inline
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # nlp-pipeline/ root, so `config` and `src` import

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import config
from src import cleaning_rules as rules
from src.io_utils import load_export

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# CDG palette — same greens used in the report's other figures, for visual consistency.
GREEN_DARK  = "#1B5E3A"
GREEN_MED   = "#2E7D4F"
GREEN_LIGHT = "#7FB89A"
GREY_DARK   = "#3A3A3A"
GREY_MED    = "#6B6B6B"
GREY_LIGHT  = "#EDEFEE"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": GREY_MED,
    "axes.labelcolor": GREY_DARK,
    "text.color": GREY_DARK,
    "xtick.color": GREY_DARK,
    "ytick.color": GREY_DARK,
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.titlecolor": GREY_DARK,
})

def pct(part, total):
    return 100 * part / total if total else 0.0

def savefig(fig, name):
    """Every figure is also saved to outputs/figures/, ready to insert in Word."""
    path = config.FIGURES / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    print(f"Figure enregistrée : {path}")


## 1. Chargement de l'export brut
*[Chapitre 4.1 — Données sources et architecture du pipeline]*

Convertit le `.xls` en parquet au premier lancement (peut prendre plusieurs
minutes sur le fichier complet), puis relit directement le parquet ensuite.


In [ ]:
df_raw = load_export()
n_raw = len(df_raw)
print(f"{n_raw:,} lignes  x  {df_raw.shape[1]} colonnes")
df_raw.head(3)


## 2. Diagnostic des colonnes — vides, constantes, identifiants
*[Chapitre 4.1]*

Reproduit `step0_diagnostic.py` sous forme de graphique : le taux de
remplissage de chaque colonne, avec les colonnes inutiles (vides, constantes,
assimilables à un identifiant) mises en évidence en gris.


In [ ]:
fill_rate = (df_raw.notna().mean() * 100).sort_values()

useless_columns = set(rules.CONCLUSION_COLUMNS) | {
    "OBSERVATION", "DEGRE_URGENCE", "ORGANISMEEMPLOYEUR", "COMMENTAIRE",
    "REPONSE_ENGAGEMENT", "CONCLUSION_VALIDATION", "DATE_NOTIFICATION", "ENG_DATE",
    "STATUT_ENGAGEMENT", "ENG_ETAT", "ENG_ACTEUR", "ETATENGAGEMENT",
    "IDENTIFIANT_CANAL", "NUMEROREQUETE", "IDT_CLIENT",
}

colors = [GREY_MED if c in useless_columns else GREEN_MED for c in fill_rate.index]

fig, ax = plt.subplots(figsize=(9, 12))
ax.barh(fill_rate.index, fill_rate.values, color=colors)
ax.set_xlabel("Taux de remplissage (%)")
ax.set_title(f"Taux de remplissage des {df_raw.shape[1]} colonnes de l'export")
ax.set_xlim(0, 100)
ax.axvline(5, color=GREY_DARK, linewidth=0.8, linestyle="--")
ax.text(5.5, 0, "5 %", fontsize=8, color=GREY_DARK, va="bottom")

handles = [
    plt.Rectangle((0, 0), 1, 1, color=GREEN_MED, label="Colonne conservée"),
    plt.Rectangle((0, 0), 1, 1, color=GREY_MED, label="Colonne écartée (vide, redondante, identifiant)"),
]
ax.legend(handles=handles, loc="lower right", frameon=False)

plt.tight_layout()
savefig(fig, "diag_fill_rate")
plt.show()

print(f"\n{len(useless_columns)} colonnes écartées sur {df_raw.shape[1]}.")


## 3. Doublons
*[Chapitre 4.1]*


In [ ]:
exact_duplicates = df_raw.duplicated().sum()
after_exact = df_raw[~df_raw.duplicated()]
conflicting_ids = int(after_exact["NUMEROREQUETE"].duplicated(keep=False).sum()) \
    if "NUMEROREQUETE" in df_raw.columns else None

print(f"Lignes strictement identiques (toutes colonnes) : {exact_duplicates:,}")
if conflicting_ids is not None:
    print(f"Lignes partageant un NUMEROREQUETE avec un contenu différent : {conflicting_ids:,}"
          f"  ({pct(conflicting_ids, n_raw):.2f}% de l'export)")


## 4. Actes automatiques vs réclamations réelles
*[Chapitre 4.1 — justification du filtrage]*

La majorité des lignes de cet export ne sont pas des réclamations : ce sont des
attestations que le client télécharge lui-même depuis le portail, enregistrées
dans la même table avec une phrase fixe en guise de description.


In [ ]:
automated_mask = df_raw["DESCRIPTION"].str.contains(
    "|".join(config.AUTOMATED_ACT_PATTERNS), case=False, na=False, regex=True
)
n_automated = int(automated_mask.sum())
n_real = n_raw - n_automated

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Donut: automated vs real ---
ax1.pie(
    [n_automated, n_real],
    labels=[f"Actes automatiques\n{n_automated:,} ({pct(n_automated, n_raw):.1f}%)",
            f"Réclamations réelles\n{n_real:,} ({pct(n_real, n_raw):.1f}%)"],
    colors=[GREY_MED, GREEN_DARK],
    startangle=90,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
    textprops={"fontsize": 10},
)
ax1.set_title(f"Répartition de l'export\n({n_raw:,} lignes)")

# --- Top 10 most frequent DESCRIPTION values ---
top10 = df_raw["DESCRIPTION"].value_counts().head(10).sort_values()
labels = [str(v)[:40] + ("…" if len(str(v)) > 40 else "") for v in top10.index]
bar_colors = [GREY_MED if any(p.lower() in str(v).lower() for p in config.AUTOMATED_ACT_PATTERNS)
              else GREEN_MED for v in top10.index]

ax2.barh(labels, top10.values, color=bar_colors)
ax2.set_xlabel("Occurrences")
ax2.set_title("10 descriptions les plus fréquentes")
for i, v in enumerate(top10.values):
    ax2.text(v, i, f"  {v:,}", va="center", fontsize=8.5, color=GREY_DARK)

plt.tight_layout()
savefig(fig, "diag_actes_automatiques")
plt.show()


## 5. Du fichier brut au corpus exploitable
*[Tableau 1 du rapport, chapitre 4.1]*


In [ ]:
office_artefacts = df_raw["DESCRIPTION"].astype(str).apply(rules.is_office_artefact)

funnel_steps = [
    ("Export brut", n_raw),
    ("- pièces jointes corrompues", n_raw - int(office_artefacts.sum())),
]
step2_n = funnel_steps[-1][1]
funnel_steps.append(("- actes automatiques", step2_n - n_automated))
step3_n = funnel_steps[-1][1]

too_short = df_raw.loc[~automated_mask & ~office_artefacts, "DESCRIPTION"] \
    .fillna("").str.len() < config.MIN_DESCRIPTION_CHARS
funnel_steps.append(("- descriptions trop courtes", step3_n - int(too_short.sum())))

labels = [s[0] for s in funnel_steps]
values = [s[1] for s in funnel_steps]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=[GREY_DARK, GREEN_LIGHT, GREEN_MED, GREEN_DARK])
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,}\n({pct(v, n_raw):.1f}%)",
             ha="center", va="bottom", fontsize=9.5)
ax.set_ylabel("Nombre de lignes")
ax.set_title("Réduction du corpus par étape de filtrage")
plt.xticks(rotation=15, ha="right")
ax.set_ylim(0, n_raw * 1.12)

plt.tight_layout()
savefig(fig, "diag_entonnoir_corpus")
plt.show()

print(f"\nCorpus exploitable estimé : {values[-1]:,} réclamations "
      f"({pct(values[-1], n_raw):.1f}% de l'export)")
print("(Le chiffre exact, après nettoyage complet des descriptions vidées, vient de step2.)")


## 6. Distribution des catégories de réclamation
*[Chapitre 4.1 — la cible `REQUEST_CATEGORY`]*


In [ ]:
categories = df_raw.loc[~automated_mask, "LIBELLE"].value_counts()

fig, ax = plt.subplots(figsize=(9, 8))
top20 = categories.head(20).sort_values()
ax.barh(top20.index, top20.values, color=GREEN_MED)
ax.set_xlabel("Nombre de réclamations")
ax.set_title(f"Top 20 des catégories (sur {categories.nunique()} au total)")
for i, v in enumerate(top20.values):
    ax.text(v, i, f"  {v:,}", va="center", fontsize=8, color=GREY_DARK)

plt.tight_layout()
savefig(fig, "diag_categories_top20")
plt.show()

print(f"Nombre total de catégories distinctes : {categories.nunique()}")
print(f"Catégories avec moins de {config.MIN_SAMPLES_PER_CLASS} exemples : "
      f"{(categories < config.MIN_SAMPLES_PER_CLASS).sum()} (seront écartées)")


## 7. Niveau de traitement (FO / MO / BO)
*[Chapitre 4.1 — la cible `ROUTING_LEVEL`, lien direct avec le workflow BPMN du volet 2]*


In [ ]:
ROUTING_LEVELS = {1: "Front-Office", 2: "Middle-Office", 3: "Back-Office"}
routing = df_raw.loc[~automated_mask, "NIVEAU_TRAITEMENT"].map(ROUTING_LEVELS).value_counts()
routing = routing.reindex(["Front-Office", "Middle-Office", "Back-Office"])

fig, ax = plt.subplots(figsize=(7, 5.5))
bars = ax.bar(routing.index, routing.values, color=[GREEN_LIGHT, GREEN_MED, GREEN_DARK])
for bar, v in zip(bars, routing.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f"{v:,}\n({pct(v, routing.sum()):.1f}%)",
             ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Nombre de réclamations")
ax.set_title("Répartition par niveau de traitement")
ax.set_ylim(0, routing.max() * 1.15)

plt.tight_layout()
savefig(fig, "diag_niveau_traitement")
plt.show()


## 8. Effet du nettoyage sur le texte
*[Chapitre 4.2 — nettoyage par règles]*

Compare la longueur des descriptions avant et après le nettoyage par
expressions régulières, sur un échantillon des réclamations réelles.


In [ ]:
sample = df_raw.loc[~automated_mask, "DESCRIPTION"].dropna().sample(
    n=min(5000, (~automated_mask).sum()), random_state=42
)
cleaned_sample = sample.apply(rules.clean_description)

fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(0, 400, 40)
ax.hist(sample.str.len().clip(upper=400), bins=bins, alpha=0.75, color=GREY_MED, label="Avant nettoyage")
ax.hist(cleaned_sample.str.len().clip(upper=400), bins=bins, alpha=0.75, color=GREEN_DARK, label="Après nettoyage")
ax.set_xlabel("Longueur de la description (caractères)")
ax.set_ylabel("Nombre de réclamations")
ax.set_title("Distribution de la longueur des descriptions")
ax.legend(frameon=False)

plt.tight_layout()
savefig(fig, "diag_longueur_avant_apres")
plt.show()

print(f"Longueur moyenne avant : {sample.str.len().mean():.0f} caractères")
print(f"Longueur moyenne après : {cleaned_sample.str.len().mean():.0f} caractères")
print(f"Réduction : {100 * (1 - cleaned_sample.str.len().mean() / sample.str.len().mean()):.1f}%")


### Exemple concret avant / après


In [ ]:
examples = sample[sample.str.len() > 80].sample(3, random_state=1)
for original in examples:
    print("AVANT :", original[:200])
    print("APRÈS :", rules.clean_description(original)[:200])
    print("-" * 100)


## 9. Illustration de la normalisation sémantique (LLM)
*[Chapitre 4.2 — apport du nettoyage LLM]*

`CONCLUSION_BO` contient plusieurs graphies différentes pour un même
événement : la normalisation sémantique doit les fusionner en une seule
classe. Ce graphique montre concrètement ce que le LLM doit accomplir.


In [ ]:
variants = ["MAJ EFFECTUEE", "maj effectue ", "mise a jour effectuée",
            "mise à jour effectuée", "MAJ EFFECTUE ", "MAJ effectuée",
            "maj effectue", "mise à jour effectuée"]

if "CONCLUSION_BO" in df_raw.columns:
    bo = df_raw["CONCLUSION_BO"].dropna().astype(str)
    counts = bo.value_counts()
    found = {v: int(counts.get(v, 0)) for v in dict.fromkeys(variants)}
    found = {k: v for k, v in found.items() if v > 0}
else:
    found = {}

if found:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [2, 1]})

    labels = list(found.keys())
    values = list(found.values())
    ax1.barh(labels, values, color=GREY_MED)
    ax1.set_xlabel("Occurrences")
    ax1.set_title("Avant normalisation — plusieurs graphies")
    for i, v in enumerate(values):
        ax1.text(v, i, f"  {v:,}", va="center", fontsize=8.5)

    total = sum(values)
    ax2.bar(["mise à jour\neffectuée"], [total], color=GREEN_DARK, width=0.5)
    ax2.text(0, total, f"{total:,}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax2.set_title("Après normalisation LLM\n(une seule classe)")
    ax2.set_ylim(0, total * 1.2)
    ax2.set_xticks([])

    plt.tight_layout()
    savefig(fig, "diag_normalisation_llm_exemple")
    plt.show()

    print(f"{len(found)} graphies distinctes -> {total:,} occurrences fusionnées en 1 classe")
else:
    print("CONCLUSION_BO absente ou ne contient pas ces variantes sur ce fichier.")


## 10. Résultats de classification
*[Chapitre 5 — à exécuter après `pipeline/step4_train.py`]*

Cette section se remplit automatiquement une fois `step4_train.py` exécuté :
elle relit les figures et métriques déjà produites, sans relancer
l'entraînement (qui prend du temps).


In [ ]:
ablation_path = config.METRICS / "step4_ablation.json"

if ablation_path.exists():
    ablation = json.loads(ablation_path.read_text(encoding="utf-8"))
    ablation_df = pd.DataFrame(ablation)
    display(ablation_df)

    for target in ablation_df["target"].unique():
        fig_path = config.FIGURES / f"ablation_{target}.png"
        if fig_path.exists():
            print(f"\n--- {target} ---")
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.imshow(plt.imread(fig_path))
            ax.axis("off")
            plt.show()
else:
    print("Pas encore de résultats : lancez d'abord")
    print("  python pipeline/step4_train.py")
    print("puis relancez cette cellule (Kernel > Restart & Run All).")


In [ ]:
for target in config.TARGETS:
    fig_path = config.FIGURES / f"confusion_{target}.png"
    if fig_path.exists():
        fig, ax = plt.subplots(figsize=(9, 8))
        ax.imshow(plt.imread(fig_path))
        ax.axis("off")
        plt.show()


## Résumé chiffré — à copier dans le rapport ou les diapositives


In [ ]:
summary = {
    "Lignes dans l'export brut": f"{n_raw:,}",
    "Actes automatiques écartés": f"{n_automated:,} ({pct(n_automated, n_raw):.1f}%)",
    "Doublons exacts": f"{exact_duplicates:,}",
    "Catégories distinctes (REQUEST_CATEGORY)": categories.nunique(),
    "Réduction moyenne de la longueur du texte": f"{100 * (1 - cleaned_sample.str.len().mean() / sample.str.len().mean()):.1f}%",
}

for label, value in summary.items():
    print(f"{label:<55} {value}")

print(f"\nToutes les figures ont été enregistrées dans : {config.FIGURES}")
